# Turtle Breakout 16 — the turtle on futures (its native market), each contract in its own account

Pre-registration: `discussions/Turtle_Breakout/018_claude_tb16_preregistration.md`. Code: `turtle_stock_accounts.py`, `turtle_stocks.py` (unchanged); universe `16/16_universe.csv`; data `tools/download_tb16_futures.py` (Yahoo, downloaded on the first run if missing).

**FUT (decision):** 41 continuous front-month futures (energy, metals, grains, softs, livestock, rates, FX, equity indices), 2000 → last session, each in its own account with the TB 01 rules (2% of HWM per trade, ≤ 3× notional), long and long + reverse; 2 bp per side, no borrow. **ETF (reported):** 18 futures-proxy ETFs, 5 bp, borrow 0.5%/yr.
Each instrument uses its own trading calendar (sessions as consecutive periods), O→H→L→C fills.
**G1** pooled mean R of real longs − random-entry longs (50 runs) ≥ 0.10R **and** per-contract differences above the calibrated fill artefact (+0.045R), t-test p < 0.025. **G2** contracts with Calmar(long + reverse) > Calmar(long): sign test p < 0.025.

Environment: `TB16_DATA` (default `<parent of DERIBIT_DATA_ROOT>/Stocks/tb16/futures_daily.parquet`), `TB16_SYNTHETIC=1`.

In [ ]:
import os, sys, json, gc, time, subprocess, tempfile, warnings, platform
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_sizing as ts, turtle_reverse as tr, turtle_portfolio as tp, turtle_stocks as tk, turtle_stock_accounts as ta

SYNTHETIC = os.getenv('TB16_SYNTHETIC', '0') == '1'
UNI = pd.read_csv(ROOT / 'notebooks' / 'Turtle_Breakout' / '16' / '16_universe.csv')
COSTS = {'fut': {'side_cost': 0.0002, 'borrow': 0.0}, 'etf': {'side_cost': 0.0005, 'borrow': 0.005}}
FUT_COST_HIGH = 0.0005
AP0 = replace(ta.AParams(), status='PREREGISTERED_TB018', seed=20261028)
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb16-')) / '16'; AP0 = replace(AP0, n_null=10)
    UNI = UNI.groupby('set').head(8).reset_index(drop=True)
    rows = []
    for i, t in enumerate(UNI.ticker):
        d, _ = tk.synthetic_daily(2600, seed=70 + i, drift=0.0001)
        rows.append(d.assign(ticker=t).rename_axis('date').reset_index())
    RAW = pd.concat(rows, ignore_index=True)
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '16'
    root = Path(os.environ['DERIBIT_DATA_ROOT']).parent
    DATA_FILE = Path(os.environ.get('TB16_DATA', str(root / 'Stocks' / 'tb16' / 'futures_daily.parquet')))
    if not DATA_FILE.exists():
        sys.path.insert(0, str(ROOT / 'tools')); import download_tb16_futures as dl
        DATA_FILE = dl.download(DATA_FILE.parent)
    RAW = pd.read_parquet(DATA_FILE)
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_turtle_stock_accounts.py'], cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
RAW['date'] = pd.to_datetime(RAW['date']).dt.tz_localize(None).dt.normalize()
RAW = RAW[RAW.date >= pd.Timestamp('2000-01-03')]
print('rows', len(RAW), '| tickers', RAW.ticker.nunique(), 'of', len(UNI), '| missing:', sorted(set(UNI.ticker) - set(RAW.ticker)))
print(json.dumps({**ta.registration(AP0), 'costs': COSTS, 'fut_cost_high': FUT_COST_HIGH}, indent=1))

## 1. Every contract / ETF in its own account

In [ ]:
def gaps(d):
    pc = d.close.shift(1); tr_ = pd.concat([d.high - d.low, (d.high - pc).abs(), (d.low - pc).abs()], axis=1).max(axis=1)
    atr = tr_.ewm(alpha=1 / 20, adjust=False).mean().shift(1)
    return int(((d.open - pc).abs() > 3 * atr).sum())

ROWS, CURVES = [], {s: {'long': {}, 'long_rev': {}, 'hold': {}, 'long_costhigh': {}, 'long_rev_costhigh': {}} for s in COSTS}
NULL = {s: [np.zeros(AP0.n_null), np.zeros(AP0.n_null)] for s in COSTS}; REAL = {s: [0.0, 0] for s in COSTS}
rng = np.random.default_rng(AP0.seed); T0 = time.time()
for r in UNI.itertuples():
    t, S = r.ticker, r.set
    AP = replace(AP0, **COSTS[S])
    P = replace(tb.Params(), min_bar_share=0.0, funding=False, fee=AP.side_cost - tb.Params().slip)
    RP = replace(tr.RParams(), side_cost=AP.side_cost)
    d = RAW[RAW.ticker == t].set_index('date')[['open', 'high', 'low', 'close']].sort_index()
    d = d[~d.index.duplicated()].dropna(); d = d[(d > 0).all(axis=1)]
    if len(d) < 250:
        ROWS.append({'ticker': t, 'set': S, 'sector': r.sector, 'status': 'too short' if len(d) else 'missing'}); continue
    SES = pd.DatetimeIndex(d.index)                     # own trading calendar
    D = tb.prepare_asset(t, tk.pseudo_bars(d, SES), None, P)
    start = D.end[0] + pd.Timedelta(days=30)
    L0 = ts.trade_list(D, P, start, RP, with_reverse=False)
    if not len(L0):
        ROWS.append({'ticker': t, 'set': S, 'sector': r.sector, 'status': 'no trades'}); continue
    L1 = ta.borrow_list(ts.trade_list(D, P, start, RP, with_reverse=True), D, AP.borrow, SES)
    a, _, sa = ta.account(D, L0, P, AP, start, SES); b, _, _ = ta.account(D, L1, P, AP, start, SES)
    first = a.index[0]; c = d.close[d.index >= first]; h = P.start_equity * c / c.iloc[0]
    for k_, v in (('long', a), ('long_rev', b), ('hold', h)):
        CURVES[S][k_][t] = v
    if S == 'fut':
        L0h, L1h = L0.assign(cost=FUT_COST_HIGH), L1.assign(cost=FUT_COST_HIGH)
        CURVES[S]['long_costhigh'][t] = ta.account(D, L0h, P, AP, start, SES)[0]; CURVES[S]['long_rev_costhigh'][t] = ta.account(D, L1h, P, AP, start, SES)[0]
    ma, mb, mh = tk.metrics(a, P.start_equity), tk.metrics(b, P.start_equity), tk.metrics(h, P.start_equity)
    Rl = ta.trade_R(L0[~L0.end], D); R1 = ta.trade_R(L1, D); Rr = R1[(L1.dir == -1).to_numpy()]
    _, T, st = tb.simulate({t: D}, P, start=start)
    nul = ta.random_long_R(D, P, start, st['p_entry'][t], AP.n_null, rng)
    REAL[S][0] += Rl.sum(); REAL[S][1] += len(Rl); NULL[S][0] += nul[:, 0]; NULL[S][1] += nul[:, 1]
    row = {'ticker': t, 'set': S, 'sector': r.sector, 'name': r.name, 'status': 'ok', 'first': first.date(), 'years': ma['years'],
           'big_gaps': gaps(d), 'longs': len(Rl), 'reverses': len(Rr), 'mean_R_long': Rl.mean(), 'mean_R_reverse': Rr.mean() if len(Rr) else np.nan,
           'null_mean_R': nul[:, 0].sum() / max(nul[:, 1].sum(), 1), 'win_rate_long': (Rl > 0).mean(), 'share_R_ge_3': (Rl >= 3).mean()}
    for pre, m in (('L', ma), ('LR', mb), ('BH', mh)):
        row.update({f'{pre}_cagr': m['cagr'], f'{pre}_max_dd': m['max_dd'], f'{pre}_calmar': m['calmar'], f'{pre}_sharpe': m['sharpe'], f'{pre}_ruined': m['ruined']})
    row['L_mean_gross'] = sa['mean_gross_leverage']
    ROWS.append(row); print(f"{t:6s} {S}: {len(Rl)} longs R {Rl.mean():+.3f} (random {row['null_mean_R']:+.3f}), {len(Rr)} reverses R {row['mean_R_reverse']:+.3f}  ({time.time() - T0:.0f}s)", flush=True)
    del D; gc.collect()
TAB = pd.DataFrame(ROWS); FUT = TAB[(TAB.set == 'fut') & (TAB.status == 'ok')]; ETF = TAB[(TAB.set == 'etf') & (TAB.status == 'ok')]
print(TAB.groupby(['set', 'status']).size())

## 2. Decision (FUT)

In [ ]:
AP = replace(AP0, **COSTS['fut'])
DEC = ta.decide_paired(REAL['fut'][0], REAL['fut'][1], NULL['fut'][0], NULL['fut'][1], FUT.mean_R_long.to_numpy(float), FUT.null_mean_R.to_numpy(float),
                       FUT.L_calmar.to_numpy(float), FUT.LR_calmar.to_numpy(float), AP)
DEC_ETF = ta.decide_paired(REAL['etf'][0], REAL['etf'][1], NULL['etf'][0], NULL['etf'][1], ETF.mean_R_long.to_numpy(float), ETF.null_mean_R.to_numpy(float),
                           ETF.L_calmar.to_numpy(float), ETF.LR_calmar.to_numpy(float), AP)
print('FUT', json.dumps(DEC, indent=1, default=float)); print('ETF (reported)', json.dumps(DEC_ETF, indent=1, default=float))

## 3. Reported: per contract, distributions, sectors, costs, average of accounts

In [ ]:
cols = ['ticker', 'sector', 'years', 'big_gaps', 'longs', 'mean_R_long', 'null_mean_R', 'win_rate_long', 'share_R_ge_3', 'mean_R_reverse',
        'L_cagr', 'L_max_dd', 'L_calmar', 'LR_cagr', 'LR_max_dd', 'LR_calmar', 'BH_cagr', 'BH_max_dd', 'BH_calmar']
display(FUT[cols].sort_values(['sector', 'ticker']).round(3)); display(ETF[cols].sort_values(['sector', 'ticker']).round(3))
q = lambda s: pd.Series({'p10': s.quantile(.1), 'p25': s.quantile(.25), 'median': s.median(), 'p75': s.quantile(.75), 'p90': s.quantile(.9), 'mean': s.mean()})
DIST = pd.concat({f'{S}_{pre}_{m}': q(X[f'{pre}_{m}']) for S, X in (('fut', FUT), ('etf', ETF)) for m in ('cagr', 'max_dd', 'calmar', 'sharpe') for pre in ('L', 'LR', 'BH')}, axis=1).T
display(DIST.round(3))
def summ(X, dec):
    return pd.Series({'instruments': len(X), 'trades long': int(X.longs.sum()), 'mean R long (pooled)': dec['real_mean_R'], 'mean R random (pooled)': dec['null_mean_R'],
                      'diff R': dec['diff_R'], 'mean R reverse (mean)': X.mean_R_reverse.mean(), 'share long CAGR > 0': (X.L_cagr > 0).mean(),
                      'share long Calmar > hold': (X.L_calmar > X.BH_calmar).mean(), 'share long+rev Calmar > long': (X.LR_calmar > X.L_calmar).mean(),
                      'share real R > own random': (X.mean_R_long > X.null_mean_R).mean(),
                      'p sign (per-instrument timing)': ta.sign_test(int((X.mean_R_long > X.null_mean_R).sum()), int(X.mean_R_long.notna().sum())),
                      'ruined long': int(X.L_ruined.sum()), 'ruined long+rev': int(X.LR_ruined.sum()), 'median mean gross (long)': X.L_mean_gross.median()})
SUMM = pd.DataFrame({'fut': summ(FUT, DEC), 'etf': summ(ETF, DEC_ETF)}); display(SUMM.round(4))
SECT = FUT.groupby('sector').agg(n=('ticker', 'size'), mean_R_long=('mean_R_long', 'mean'), null_mean_R=('null_mean_R', 'mean'), mean_R_reverse=('mean_R_reverse', 'mean'),
                                 L_cagr=('L_cagr', 'median'), LR_cagr=('LR_cagr', 'median'), BH_cagr=('BH_cagr', 'median'), L_max_dd=('L_max_dd', 'median'), LR_max_dd=('LR_max_dd', 'median'))
display(SECT.round(3))
POOL = {}
for S in COSTS:
    for v, cur in CURVES[S].items():
        if cur:
            POOL[(S, v)] = 1e5 * np.exp(tk.equal_weight_hold(cur).cumsum())
AVG = pd.DataFrame({f'{S}|{v}': tk.metrics(s, 1e5) for (S, v), s in POOL.items()}).T
print('Average of the per-instrument accounts (each 1/N; not a portfolio test)'); display(AVG.round(3))

## 4. Figures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
x = FUT.sort_values('mean_R_long'); y = np.arange(len(x))
axes[0].barh(y, x.mean_R_long, color='#1f5aa6', alpha=0.8, label='real breakouts'); axes[0].scatter(x.null_mean_R, y, color='k', s=12, zorder=3, label='random entries')
axes[0].set_yticks(y); axes[0].set_yticklabels(x.ticker, fontsize=7); axes[0].axvline(0, color='k', lw=0.6); axes[0].set_xlabel('mean net R per long trade'); axes[0].legend(frameon=False, fontsize=8)
lim = (-0.5, 1.5)
axes[1].scatter(FUT.L_calmar.clip(*lim), FUT.LR_calmar.clip(*lim), s=14); axes[1].plot(lim, lim, color='k', lw=0.8)
axes[1].set_xlabel('Calmar long'); axes[1].set_ylabel('Calmar long + reverse'); axes[1].set_title(f"G2: {DEC['reverse_better']}/{DEC['instruments']} above the line", loc='left', fontsize=10)
for v, col in (('hold', '0.6'), ('long', '#1f5aa6'), ('long_rev', '#b33f40')):
    s = POOL[('fut', v)]; axes[2].plot(s.index, s / 1e5, color=col, label=v)
axes[2].set_yscale('log'); axes[2].legend(frameon=False); axes[2].grid(alpha=0.25); axes[2].set_title('FUT: average of per-contract accounts', loc='left', fontsize=10)
plt.tight_layout(); plt.show()

## 5. Save

In [ ]:
TAB.to_csv(OUT / '16_per_instrument.csv', index=False); DIST.to_csv(OUT / '16_distributions.csv'); SUMM.to_csv(OUT / '16_summary.csv')
SECT.to_csv(OUT / '16_sectors.csv'); AVG.to_csv(OUT / '16_average_of_accounts.csv')
(OUT / '16_decision.json').write_text(json.dumps({'decision_fut': DEC, 'etf_reported': DEC_ETF, 'registration': {**ta.registration(AP0), 'costs': COSTS},
                                                   'synthetic': SYNTHETIC}, indent=1, default=float))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '16_manifest.json').write_text(json.dumps({'version': ta.VERSION, 'git': git, 'python': platform.python_version(),
    'created_utc': pd.Timestamp.now('UTC').isoformat(), 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print('VERDICT (FUT):', DEC['verdict'])